# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ayaahmed571/Flyrank-ml-internship/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

I chose a Decision Tree because it fits my lane of identifying content pages that may need review or refresh.

The tree provides an interpretable model that can use several signals together instead of relying on the fixed thresholds in my Week-4 baseline. This makes it possible to compare whether a learned pattern provides better decision support than the simple rule.

I chose interpretability over unnecessary complexity because the goal is to understand and prioritize pages, not just maximize model complexity.

## 2. Split design

I use a client-grouped train/test split so that pages from the same client do not appear in both training and test data.

This is a more honest evaluation because the model is tested on clients whose pages it did not see during training. I use the same target and evaluation metric for the model and the Week-4 baseline so the comparison is fair.

In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

print(df.shape)
print(df.columns.tolist())

df["is_declining_label"] = (df["trend_direction"] == "down").astype(int)

print(df["is_declining_label"].value_counts())

from sklearn.model_selection import GroupShuffleSplit

features = [
    "content_age_days",
    "days_since_last_update",
    "impressions_90d",
    "avg_position",
    "ctr",
    "word_count"
]

X = df[features].copy()
y = df["is_declining_label"]
groups = df["client_id"]

gss = GroupShuffleSplit(
    n_splits=1,
    test_size=0.2,
    random_state=42
)

train_idx, test_idx = next(
    gss.split(X, y, groups=groups)
)

X_train = X.iloc[train_idx]
X_test = X.iloc[test_idx]

y_train = y.iloc[train_idx]
y_test = y.iloc[test_idx]

print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)

print(
    "Clients in both train and test:",
    len(
        set(groups.iloc[train_idx])
        & set(groups.iloc[test_idx])
    )
)

(30000, 44)
['content_id', 'client_id', 'search_volume', 'competition', 'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count', 'char_count', 'provider_used', 'model_used', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'age_tier', 'age_tier_order', 'days_since_last_update', 'freshness_tier', 'word_count_tier', 'char_count_tier', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'impression_tier', 'position_tier', 'trend_direction', 'trend_pct']
is_declining_label
1    16262
0    13738
Name: count, dtype: int64
Train shape: (23837, 6)
Test shape: (6163, 6)
Clients in both train and test: 0


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

In [ ]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import precision_score

# Handle missing / infinite values
X_train_clean = X_train.replace([np.inf, -np.inf], np.nan).fillna(0)
X_test_clean = X_test.replace([np.inf, -np.inf], np.nan).fillna(0)

# Train model
model = DecisionTreeClassifier(
    max_depth=3,
    class_weight="balanced",
    random_state=42
)

model.fit(X_train_clean, y_train)

# Predictions
y_pred = model.predict(X_test_clean)

# Model metric
model_precision = precision_score(
    y_test,
    y_pred,
    zero_division=0
)

print(f"Decision Tree Precision: {model_precision:.3f}")

Decision Tree Precision: 0.579


In [ ]:
baseline_test_score = (
    (X_test["days_since_last_update"] >= 180).astype(int) * 2
    + (X_test["impressions_90d"] >= 500).astype(int)
)

baseline_order = np.argsort(-baseline_test_score.values)

n_predicted_positive = y_pred.sum()

baseline_top = y_test.iloc[
    baseline_order[:n_predicted_positive]
]

baseline_precision = baseline_top.mean()

print(f"Decision Tree Precision: {model_precision:.3f}")
print(f"Baseline Precision:      {baseline_precision:.3f}")

Decision Tree Precision: 0.579
Baseline Precision:      0.509


### Model vs Baseline

On the held-out client-grouped test set, the Decision Tree achieved a measured Precision of 0.579, compared with 0.509 for the Week-4 baseline rule.

This is a directional result suggesting that the Decision Tree may improve the prioritization of potentially declining pages compared with the fixed rule. However, this result alone does not establish that the model will generalize to new clients.

## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

In [ ]:
errors = df.iloc[test_idx].copy()

errors["actual"] = y_test.values
errors["predicted"] = y_pred

# False positives: model predicted declining, but actual was not declining
false_positives = errors[
    (errors["predicted"] == 1) &
    (errors["actual"] == 0)
]

# False negatives: model missed a declining page
false_negatives = errors[
    (errors["predicted"] == 0) &
    (errors["actual"] == 1)
]

print("False positives:", len(false_positives))
print("False negatives:", len(false_negatives))

False positives: 1248
False negatives: 1434


In [ ]:
from sklearn.inspection import permutation_importance

perm = permutation_importance(
    model,
    X_test_clean,
    y_test,
    n_repeats=10,
    random_state=42,
    scoring="precision"
)

importance = pd.Series(
    perm.importances_mean,
    index=features
).sort_values(ascending=False)

print(importance)

impressions_90d           0.044074
content_age_days          0.041663
days_since_last_update    0.000000
avg_position              0.000000
ctr                       0.000000
word_count                0.000000
dtype: float64


In [ ]:
comparison = pd.DataFrame({
    "Method": ["Week-4 Baseline", "Decision Tree"],
    "Precision": [baseline_precision, model_precision]
})

comparison

,Method,Precision
0,Week-4 Baseline,0.508944
1,Decision Tree,0.578805


The Decision Tree achieved a measured Precision of 0.579 on the held-out client-grouped test set, compared with 0.509 for the Week-4 baseline.

The model therefore showed a higher measured precision on this split. This is a directional result and should be validated further before claiming that the model will generalize to unseen clients.

## Self-check

Before you submit, confirm each line honestly:

- [X] Every section above is filled — markdown thinking AND the code that backs it
- [X] The notebook runs top to bottom with no errors (Runtime → Run all)
- [X] No client names, URLs, or private queries anywhere
- [X] My claims use careful words: observed, measured, directional, decision-support
- [X] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.